# 14장 한 파일에서 패키지로 — 코드 구조화

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch14_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | 열 개 파일을 구조대로 만들어 치고, `--run mlp-e2` 로 실행해 13장과 숫자 대조 | [ch14_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch14_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | 각 `__init__.py` 에 "이 패키지가 밖에 노출하는 것"을 주석으로 | [ch14_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch14_2_comment.ipynb) |
| 3. 다시 만들기 | 13장 `train.py` 를 새 폴더에 복사한 뒤 이 장을 보지 않고 세 패키지로 나누기. 14.3의 표를 직접 만들기 | [ch14_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch14_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "14"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/14")

준비 끝. 작업 폴더: code/14


## 파일 구조

13장의 한 파일이 열 개 파일로 나뉘었습니다.

In [ ]:
from pathlib import Path
for p in sorted(Path('.').rglob('*.py')):
    print(p.as_posix())

data/__init__.py
data/mnist.py
infer.py
networks/__init__.py
networks/mlp.py
train.py
utils/__init__.py
utils/checkpoint.py
utils/metrics.py
utils/parsing.py


## 부품 — 세 패키지

## `networks/__init__.py` — networks 패키지: 모델 정의를 모아 둔다. 이 파일이 있어야 폴더가 패키지로 인식된다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# networks 패키지: 모델 정의를 모아 둔다. 이 파일이 있어야 폴더가 패키지로 인식된다.
from .mlp import MLP

MODELS = {"mlp": MLP}          # 이름 → 클래스. 16장부터 여기에 lenet, vgg, ... 를 추가한다


def build_model(name, **kwargs):
    """이름으로 모델을 만든다. 예: build_model('mlp', hidden=128)"""
    if name not in MODELS:
        raise ValueError(f"모르는 모델 {name!r}. 가능한 것: {list(MODELS)}")
    return MODELS[name](**kwargs)
```

## `networks/mlp.py`

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
import torch
import torch.nn as nn


class MLP(nn.Module):
    """다층 퍼셉트론. 입력을 한 줄로 펴서 두 선형 층을 지난다."""

    def __init__(self, in_dim=784, hidden=128, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))
```

## `data/__init__.py` — data 패키지: 데이터셋과 로더를 만드는 함수를 모아 둔다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# data 패키지: 데이터셋과 로더를 만드는 함수를 모아 둔다.
from .mnist import get_mnist_loaders

LOADERS = {"mnist": get_mnist_loaders}     # 15장부터 cifar10, 내 이미지 폴더 등을 추가한다


def get_loaders(name, **kwargs):
    if name not in LOADERS:
        raise ValueError(f"모르는 데이터셋 {name!r}. 가능한 것: {list(LOADERS)}")
    return LOADERS[name](**kwargs)
```

## `data/mnist.py`

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


def get_mnist_loaders(data_root, batch_size=64, test_batch_size=1000):
    """(학습 로더, 시험 로더) 를 돌려준다."""
    transform = transforms.ToTensor()
    train_set = datasets.MNIST(data_root, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(data_root, train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=test_batch_size)
    return train_loader, test_loader
```

## `utils/__init__.py` — utils 패키지: 인자 파싱, 평가, 저장처럼 모델·데이터와 무관한 도구.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# utils 패키지: 인자 파싱, 평가, 저장처럼 모델·데이터와 무관한 도구.
from .parsing import parse_train_args, parse_infer_args
from .metrics import evaluate
from .checkpoint import make_run_dir, save_json, load_json
```

## `utils/parsing.py`

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
import argparse
import os


def parse_train_args(argv=None):
    p = argparse.ArgumentParser(description="학습")
    p.add_argument("--model", default="mlp", help="networks.MODELS 의 이름")
    p.add_argument("--dataset", default="mnist", help="data.LOADERS 의 이름")
    p.add_argument("--data-root", default=os.environ.get("JUCOS_DATA", "./data"))
    p.add_argument("--results", default="results")
    p.add_argument("--run", default=None)
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--hidden", type=int, default=128)
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--log-every", type=int, default=200)
    return p.parse_args(argv)


def parse_infer_args(argv=None):
    p = argparse.ArgumentParser(description="추론")
    p.add_argument("--results", default="results")
    p.add_argument("--run", required=True)
    p.add_argument("--which", default="best", choices=["best", "last"])
    p.add_argument("--index", type=int, default=None)
    return p.parse_args(argv)
```

## `utils/metrics.py`

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
import torch


def evaluate(model, loader, loss_fn, device):
    """(정확도, 평균 손실)"""
    model.eval()
    correct, total_loss = 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(1) == labels).sum().item()
    model.train()
    n = len(loader.dataset)
    return correct / n, total_loss / n
```

## `utils/checkpoint.py`

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
import json
import os
import time


def make_run_dir(results, run=None):
    """results/{run}/ 을 만들고 경로를 돌려준다. run 이 없으면 시각으로 이름 짓는다."""
    out_dir = os.path.join(results, run or time.strftime("%Y%m%d-%H%M%S"))
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)
```

## 흐름 — `train.py` 와 `infer.py`

## `train.py` — 14장 train.py: 13장의 한 파일을 패키지로 나눈 뒤 남은 것 = 흐름(main)만.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 14장 train.py: 13장의 한 파일을 패키지로 나눈 뒤 남은 것 = 흐름(main)만.
# 실행: python train.py --model mlp --dataset mnist --epochs 2 --run mlp-e2
import os
import time

import torch
import torch.nn as nn

from networks import build_model
from data import get_loaders
from utils import parse_train_args, evaluate, make_run_dir, save_json


def main():
    args = parse_train_args()
    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = make_run_dir(args.results, args.run)
    save_json(vars(args), os.path.join(out_dir, "config.json"))

    train_loader, test_loader = get_loaders(args.dataset, data_root=args.data_root, batch_size=args.batch_size)
    model = build_model(args.model, hidden=args.hidden).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)

    history, best_acc = [], 0.0
    for epoch in range(1, args.epochs + 1):
        running, t0 = 0.0, time.time()
        for step, (images, labels) in enumerate(train_loader, start=1):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
            if step % args.log_every == 0:
                print(f"  에포크 {epoch} 걸음 {step:4d}  평균 손실 {running / args.log_every:.4f}")
                running = 0.0
        acc, test_loss = evaluate(model, test_loader, loss_fn, device)
        history.append({"epoch": epoch, "test_acc": acc, "test_loss": test_loss, "seconds": time.time() - t0})
        print(f"에포크 {epoch}: 시험 정확도 {acc:.4f}  시험 손실 {test_loss:.4f}")
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(out_dir, "model_best.pt"))
    torch.save(model.state_dict(), os.path.join(out_dir, "model_last.pt"))
    save_json({"best_test_acc": best_acc, "history": history}, os.path.join(out_dir, "metrics.json"))
    print(f"저장 완료: {out_dir}/")


if __name__ == "__main__":
    main()
```

In [ ]:
!python train.py --model mlp --dataset mnist --epochs 2 --run mlp-e2

  에포크 1 걸음  200  평균 손실 0.8553
  에포크 1 걸음  400  평균 손실 0.3859
  에포크 1 걸음  600  평균 손실 0.3291
  에포크 1 걸음  800  평균 손실 0.2942
에포크 1: 시험 정확도 0.9250  시험 손실 0.2607
  에포크 2 걸음  200  평균 손실 0.2533
  에포크 2 걸음  400  평균 손실 0.2304
  에포크 2 걸음  600  평균 손실 0.2312
  에포크 2 걸음  800  평균 손실 0.2108
에포크 2: 시험 정확도 0.9473  시험 손실 0.1804
저장 완료: results\mlp-e2/


## `infer.py` — 14장 infer.py: 패키지 구조에서의 추론. 모델 이름·설정은 config.json 에서 읽는다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 14장 infer.py: 패키지 구조에서의 추론. 모델 이름·설정은 config.json 에서 읽는다.
import os

import torch
from torchvision import datasets, transforms

from networks import build_model
from utils import parse_infer_args, load_json


def load_model(out_dir, which):
    config = load_json(os.path.join(out_dir, "config.json"))
    model = build_model(config["model"], hidden=config["hidden"])
    model.load_state_dict(torch.load(os.path.join(out_dir, f"model_{which}.pt"), map_location="cpu"))
    model.eval()
    return model, config


def main():
    args = parse_infer_args()
    out_dir = os.path.join(args.results, args.run)
    model, config = load_model(out_dir, args.which)
    print(f"모델 불러옴: {out_dir}/model_{args.which}.pt  ({config['model']}, hidden={config['hidden']})")
    test_set = datasets.MNIST(config["data_root"], train=False, download=True, transform=transforms.ToTensor())
    indices = [args.index] if args.index is not None else list(range(8))
    with torch.no_grad():
        for i in indices:
            image, label = test_set[i]
            probs = torch.softmax(model(image.unsqueeze(0)), dim=1)[0]
            pred = probs.argmax().item()
            print(f"시험 이미지 {i:5d}: 정답 {label}  예측 {pred}  확신 {probs[pred]:.3f}{'' if pred == label else '  ← 틀림'}")


if __name__ == "__main__":
    main()
```

In [ ]:
!python infer.py --run mlp-e2

모델 불러옴: results\mlp-e2/model_best.pt  (mlp, hidden=128)
시험 이미지     0: 정답 7  예측 7  확신 0.996
시험 이미지     1: 정답 2  예측 2  확신 0.984
시험 이미지     2: 정답 1  예측 1  확신 0.979
시험 이미지     3: 정답 0  예측 0  확신 0.999
시험 이미지     4: 정답 4  예측 4  확신 0.959
시험 이미지     5: 정답 1  예측 1  확신 0.990
시험 이미지     6: 정답 4  예측 4  확신 0.967
시험 이미지     7: 정답 9  예측 9  확신 0.961


## 결과 폴더

13장과 같은 파일이 생겼는지 봅니다.

In [ ]:
import os
print(sorted(os.listdir('results/mlp-e2')))

['config.json', 'metrics.json', 'model_best.pt', 'model_last.pt']


## 다음 단계

- 2단계 「주석 달기」: [ch14_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch14_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch14_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch14_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.